<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml_v2/exercices/seance1_exercices.ipynb)

# Séance 4.1 — Le Machine Learning : prédire n'est pas expliquer

**Exercices** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- définir clairement **ce que vous souhaitez prédire** et les informations utilisées pour le faire
- séparer les données entre **apprentissage** et **test** afin d'évaluer le modèle sur des observations qu'il n'a jamais vues
- entraîner un modèle puis produire des prédictions avec `fit()` et `predict()`
- mesurer la qualité des prédictions à l'aide de la **MAE**, de la **RMSE** et du **R²**
- comparer les performances du modèle à une **prédiction de référence simple**
- vérifier que le modèle généralise correctement, sans **surapprentissage** ni **fuite de données**

## Exercice — Le consultant et l'estimateur

## La question

Votre estimateur tourne depuis six mois à l'agence. Vous l'aviez construit au
bloc 3 : pour un arrondissement et une surface, il annonce une fourchette de
prix.

Un cabinet de conseil vient de passer. Il propose de le remplacer par « un
modèle de machine learning entraîné sur toutes les ventes de Paris ». Devis :
**40 000 €**.

La direction vous demande un avis. Pas une opinion : **un avis chiffré**.

> *« Est-ce que leur modèle fait mieux que le nôtre ? De combien ? Et sur
> quoi vous appuyez-vous pour le dire ? »*

À la fin de cet exercice, vous aurez entraîné ce modèle vous-même, vous
saurez exactement où il se trompe, et vous aurez écrit la réponse au
consultant.

### Les trois règles qui vont s'affronter

| | Ce qu'elle annonce | Ce qu'elle a estimé sur les données |
|---|---|---|
| **la règle paresseuse** | toujours le prix moyen | 1 nombre |
| **l'estimateur maison** | `surface × médiane du prix au m² de l'arrondissement` | 20 médianes |
| **le modèle du consultant** | une régression linéaire | 21 coefficients |

Une bonne partie du travail consistera à comprendre pourquoi **ces trois-là
se comparent**, et pourquoi le classement n'est pas celui qu'on attend.

## Comment ça marche

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* nomme les outils dont vous avez besoin.

Cet exercice est plus difficile que les précédents, et il est construit pour
que vous ne restiez jamais bloqué :

- la **mécanique** — les appels à scikit-learn qui ne s'inventent pas — vous est donnée ;
- ce que vous écrivez, ce sont les **choix** : quelles colonnes, quelle mesure, quel verdict ;
- les **cellules de pari** vous demandent d'annoncer un résultat *avant* de l'exécuter. Elles ne sont pas décoratives : c'est en se trompant de pari qu'on apprend à lire un chiffre ;
- les **cellules de vérification** affichent `OK` ou `A REVOIR` avec un indice, aux endroits où une erreur fausserait la suite.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

## Partie 0 — Mise en route

Exécutez les trois cellules suivantes. La première reprend les imports du
cours 4.1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "immo_paris_2024.csv")

print(ventes.shape)
ventes.head(3)

Le fichier des blocs 2 et 3 : 25 209 ventes d'appartements parisiens de 2024,
une ligne par vente. Vous le connaissez.

| Colonne | Contenu |
|---|---|
| `vente_id` | l'identifiant de la vente |
| `date` | la date de la vente |
| `prix` | le prix payé, en euros |
| `rue` | le nom de la rue |
| `arrondissement` | de 1 à 20 |
| `surface` | la surface, en m² |
| `pieces` | le nombre de pièces principales |
| `longitude`, `latitude` | la position sur la carte |
| `prix_m2` | le prix divisé par la surface |

---

## Partie 1 — Décider avant de coder

Un modèle ne commence pas par `.fit()`. Il commence par deux questions :
**qu'est-ce que je prédis**, et **qu'est-ce que je connaîtrai au moment où il
faudra le prédire ?**

La seconde est celle que le cours 4.1 pose pour repérer une fuite de données.
Posez-la à chaque colonne du fichier, une par une :

> *Le jour où un client m'appelle pour faire estimer son appartement, est-ce
> que je connais cette valeur ?*

### Exercice 1 — Trier les dix colonnes

Répartissez les dix noms de colonnes dans quatre listes :

- `cible` : ce qu'on cherche à prédire ;
- `interdites` : ce qui ne sera **pas connu** au moment d'estimer, ou ce qui se
  calcule à partir de la réponse ;
- `sans_interet` : ce qui ne décrit pas le bien et ne peut rien prédire ;
- `utilisables` : tout le reste.

Aucune ligne de scikit-learn dans cet exercice. C'est du raisonnement, et
c'est le plus important du notebook.

> **Rappel.** `list(ventes.columns)` vous donne les dix noms à répartir.
> Chaque nom va dans **une seule** liste.

In [ ]:
verifier("la cible", cible == ["prix"], "la question est : combien vaut cet appartement ?")
verifier("les interdites", sorted(interdites) == ["prix_m2"],
         "quelle colonne se calcule a partir du prix ? le jour de l'estimation, elle n'existe pas encore")
verifier("l'identifiant", sans_interet == ["vente_id"], "un numero de dossier ne predit rien")
verifier("les utilisables", len(utilisables) == 7, "les sept qui restent, y compris la rue et la date")

**`prix_m2` est le piège**, et il est parfait : c'est la colonne la plus
utile du fichier, celle sur laquelle repose tout l'exercice du bloc 3. Elle
est aussi strictement interdite ici, parce qu'elle se calcule en divisant le
prix par la surface. Le jour où un client appelle, **elle n'existe pas
encore**. La partie 5 montrera ce qui se passe quand on l'oublie.

### Votre fiche de décision

Complétez cette cellule de texte (double-clic pour l'éditer). Vous la
relirez à la fin.

- Ce que je prédis : …
- Ce que je connais au moment de le prédire : …
- Pour l'agence, se tromper de 50 000 € sur une estimation, c'est : …
- Donc la mesure qui m'intéresse le plus est : …

---

## Partie 2 — Le modèle du consultant

### Exercice 2 — Mettre des ventes de côté

Commençons petit : une seule variable explicative, la surface.

Construisez `X` (un tableau à une colonne, `surface`) et `y` (la colonne
`prix`), puis découpez-les en apprentissage et test avec `test_size=0.25` et
`random_state=67`.

> **Rappel.** `train_test_split(X, y, test_size=..., random_state=...)` rend
> **quatre** objets, dans l'ordre : les `X` d'abord, apprentissage puis test,
> les `y` ensuite. Un tableau à une colonne s'écrit avec deux paires de
> crochets : `ventes[["surface"]]`.

In [ ]:
verifier("le jeu d'apprentissage", len(X_train) == 18906, "test_size=0.25 : un quart part au test")
verifier("le jeu de test", len(X_test) == 6303, "random_state=67, comme dans le cours")

Ces 6 303 ventes ne serviront **qu'à la fin**, pour noter. Le
`random_state=67` est important : il fixe le tirage, donc tout le monde note
sur exactement les mêmes ventes, et les trois règles de cet exercice seront
comparées sur les mêmes aussi.

### Exercice 3 — Entraîner, prédire

Ajustez une régression linéaire sur le jeu d'apprentissage, puis produisez
les prédictions sur les deux jeux, dans `pred_train` et `pred_test`.

> **Rappel.** `LinearRegression().fit(...)` apprend, `.predict(...)` annonce.
> Le modèle n'apprend **que** sur l'apprentissage.

### Exercice 4 — La note

Avant de la calculer, engagez-vous. Écrivez votre pari dans la cellule
ci-dessous : **de combien d'euros ce modèle se trompe-t-il en moyenne ?**

In [ ]:
# Mon pari : ce modele se trompe en moyenne de ....... euros

Calculez maintenant les trois mesures du cours sur le **jeu de test** :
`mae`, `rmse` et `r2`.

> **Rappel.** `mean_absolute_error(reel, predit)`, `r2_score(reel, predit)`,
> et la RMSE est la racine carrée de `mean_squared_error(...)` — donc
> `** 0.5`.

In [ ]:
verifier("la MAE", round(mae) == 160175, "mean_absolute_error(y_test, pred_test), sur le jeu de TEST")
verifier("le R2", round(r2, 3) == 0.757, "r2_score(y_test, pred_test)")

**R² = 0,757, MAE = 160 175 €.**

Ces deux nombres décrivent le même modèle. Le premier est celui que le
consultant met sur sa première diapositive. Le second est celui que le
directeur comprend : l'estimation se trompe en moyenne de 160 175 € sur un
prix médian de 390 000 €.

Comparez-le à votre pari. Si vous aviez annoncé beaucoup moins, vous venez de
faire l'expérience la plus utile du notebook.

### Exercice 5 — Donner plus d'informations au modèle

Le modèle ne connaît que la surface. Il ignore qu'un mètre carré du 19e et un
mètre carré du 6e ne valent pas la même chose.

La cellule suivante prépare les variables : `pieces`, et l'arrondissement
transformé en colonnes de 0 et de 1. Exécutez-la.

In [ ]:
# get_dummies : une colonne 0/1 par arrondissement. Un modele multiplie des
# nombres, il ne sait pas quoi faire de "16e". drop_first : le 1er sert de
# reference, sa colonne n'apporterait rien de plus.
arrondissements = pd.get_dummies(ventes["arrondissement"], prefix="arr", drop_first=True)
X_grand = pd.concat([ventes[["surface", "pieces"]], arrondissements], axis=1)

print(X_grand.shape[1], "colonnes :", list(X_grand.columns[:4]), "...")

Refaites le découpage avec `X_grand` — **le même `random_state=67`**, sinon
les deux modèles ne seraient plus notés sur les mêmes ventes et la
comparaison ne voudrait rien dire. Entraînez, prédisez dans `pred_grand`, et
recalculez les trois mesures.

> **Rappel.** Les mêmes quatre lignes qu'aux exercices 2, 3 et 4. Nommez les
> jeux `Xg_train`, `Xg_test`, `yg_train`, `yg_test`.

In [ ]:
verifier("la MAE du grand modele", round(mae_grand) == 149483, "meme random_state=67, sinon le test change")
verifier("le R2 du grand modele", round(r2_grand, 3) == 0.791, "r2_score sur le jeu de test du grand modele")
verifier("les memes ventes de test", list(yg_test.index) == list(y_test.index),
         "si les index different, les deux modeles ne sont pas notes sur les memes ventes")

10 691 € de MAE gagnés en donnant l'arrondissement au modèle. C'est réel,
et c'est modeste : le modèle se trompe encore de 149 483 € en moyenne.

### Exercice 6 — Le coefficient qui choque

La cellule suivante affiche ce que le modèle a appris. Exécutez-la, puis
lisez la ligne `pieces`.

In [ ]:
appris = pd.Series(grand.coef_, index=X_grand.columns).round(0)

print("constante :", round(grand.intercept_), "euros")
print(appris.head(2))
print("...")
print(appris.filter(like="arr").sort_values(ascending=False).head(3))

**`pieces` : -77 924 €.** À surface égale, une pièce de plus fait
**baisser** le prix annoncé par le modèle de 77 924 €.

En commentaire, dans la cellule ci-dessous : *est-ce que ce modèle vous dit
qu'il faut abattre une cloison pour gagner 77 924 € ?* Répondez, et
expliquez ce que ce coefficient décrit vraiment.

> **Rappel.** Le titre du cours 4.1 est la réponse : prédire n'est pas
> expliquer.

---

## Partie 3 — Regarder les prédictions, pas les scores

Trois nombres résument 6 303 prédictions. C'est commode, et c'est
insuffisant. Cette partie ouvre le capot.

La cellule suivante trace le nuage du cours 4.1 : en abscisse le prix réel,
en ordonnée le prix annoncé. Sur la droite rouge, le modèle tombe juste.

In [ ]:
plt.figure(figsize=(7, 4.4))
plt.scatter(yg_test, pred_grand, s=7, alpha=0.25, color="#2878B5")
plt.plot([0, 3e6], [0, 3e6], color="#D64541", linewidth=2, label="prediction parfaite")
plt.axhline(0, color="black", linewidth=1)
plt.xlim(0, 3e6); plt.ylim(-4e5, 3e6)   ## on coupe a 3 millions : 2 % des ventes sont au-dela
plt.xlabel("prix reel (euros)"); plt.ylabel("prix annonce par le modele (euros)")
plt.title("Predit contre observe")
plt.legend()
plt.tight_layout()
plt.show()

Le nuage **s'ouvre en éventail** : sur les petits prix il colle à la droite,
et plus on monte plus il s'étale. Regardez aussi sous le trait noir, à
gauche.

### Exercice 7 — Compter l'impossible

Combien de prix annoncés par le modèle sont **négatifs** ? Mettez le compte
dans `nb_negatifs`, et affichez le plus bas.

> **Rappel.** Une comparaison sur un tableau de nombres rend des Vrai/Faux, et
> `.sum()` compte les Vrai — comme au bloc 2. `.min()` donne le plus petit.

In [ ]:
verifier("les prix negatifs", nb_negatifs == 139, "(pred_grand < 0).sum() : un True vaut 1")

**139 appartements parisiens à prix négatif**, le pire à -122 681 €. Le
modèle annonce qu'on vous paiera 122 681 € pour emporter
l'appartement.

Aucune des trois mesures ne l'avait signalé. Un R² de 0,791 est compatible
avec des prédictions absurdes, parce qu'une moyenne d'erreurs ne regarde pas
les prédictions une par une. **Vous, si.**

### Exercice 8 — L'erreur a-t-elle une forme ?

La cellule suivante construit le tableau d'analyse : pour chaque vente du
jeu de test, ce qui s'est passé, ce que le modèle annonçait, l'écart en euros
et l'écart en pourcentage. Exécutez-la.

In [ ]:
erreurs = pd.DataFrame({"reel": yg_test, "predit": pred_grand})
erreurs["surface"] = ventes.loc[erreurs.index, "surface"]
erreurs["rue"] = ventes.loc[erreurs.index, "rue"]
erreurs["arrondissement"] = ventes.loc[erreurs.index, "arrondissement"]
erreurs["ecart"] = erreurs["predit"] - erreurs["reel"]     ## positif = le modele a vu trop grand
erreurs["absolu"] = erreurs["ecart"].abs()
erreurs["pct"] = 100 * erreurs["ecart"] / erreurs["reel"]

# Six tranches de surface, des studios aux hotels particuliers
erreurs["tranche"] = pd.cut(erreurs["surface"], [0, 25, 40, 60, 90, 150, 2000],
                            labels=["moins de 25", "25 a 40", "40 a 60",
                                    "60 a 90", "90 a 150", "plus de 150"])
erreurs.head(3)

Calculez maintenant `biais` : la **médiane** de l'erreur relative (`pct`) pour
chaque tranche de surface. Affichez-la.

Pourquoi la médiane et pas la moyenne ? Vous avez la réponse depuis le
bloc 3 : quelques erreurs énormes suffisent à déplacer une moyenne.

> **Rappel.** `groupby("tranche", observed=True)` puis la colonne et
> `median()`. L'argument `observed=True` évite un avertissement quand une
> tranche est vide.

In [ ]:
verifier("le biais des studios", round(biais.loc["moins de 25"], 1) == -42.5,
         "la mediane de la colonne pct, tranche par tranche")
verifier("le biais des 60-90", round(biais.loc["60 a 90"], 1) == 12.8, "meme calcul")

| surface | moins de 25 | 25 à 40 | 40 à 60 | 60 à 90 | 90 à 150 | plus de 150 |
|---|---:|---:|---:|---:|---:|---:|
| erreur médiane | **-42,5 %** | -10,6 % | 4,7 % | **12,8 %** | 10,1 % | 4,6 % |

Ce ne sont pas des erreurs au hasard : **le modèle sous-évalue un studio sur
deux de plus de 40 %**, et surévalue les 60-90 m². Une erreur qui va toujours
dans le même sens dans un coin des données, ça porte un nom : c'est un
**biais**, et ça veut dire que la forme du modèle ne colle pas au terrain.

Retenez-le, la partie 4 va dire d'où il vient.

### Exercice 9 — Les cinq pires

Affichez les cinq ventes que le modèle rate le plus, avec leur rue, leur
surface, leur prix réel et le prix annoncé. Mettez la plus grosse erreur
absolue dans `pire_erreur`.

> **Rappel.** `df.nlargest(5, "colonne")` garde les cinq plus grandes valeurs,
> déjà triées. `.max()` sur une colonne pour la plus grande.

In [ ]:
verifier("la pire erreur", round(pire_erreur) == 4268971, "le max de la colonne absolu")

Des hôtels particuliers du 16e et du 8e, à 4 268 971 € près pour le pire. Ces
biens-là ne ressemblent à rien de ce que le modèle a vu : il les
sous-estime de plusieurs millions.

Et ils pèsent lourd dans la note. La cellule suivante le chiffre.

In [ ]:
seuil_pires = erreurs["absolu"].quantile(0.95)
pires = erreurs.query("absolu > @seuil_pires")

print(len(pires), "ventes sur", len(erreurs), "portent",
      round(100 * pires["absolu"].sum() / erreurs["absolu"].sum()), "% de l'erreur totale")
print("erreur mediane :", round(erreurs["absolu"].median()), "euros")
print("MAE            :", round(erreurs["absolu"].mean()), "euros")

**L'erreur médiane vaut 92 621 €, la MAE 149 483 €.** La moitié des
estimations tombe à moins de 92 621 € — et la moyenne est tirée vers le
haut par les 5 % du haut, qui portent 29 % de l'erreur.

### Une décision, pas un calcul

À partir de quelle surface interdiriez-vous à l'agence de se servir de cet
estimateur ? Complétez cette cellule de texte :

- Je déconseille l'estimateur au-dessus de … m², parce que …
- Ce que le commercial dit au client dans ce cas : …

La cellule suivante donne un argument de plus. Elle entraîne un modèle **sur
les seuls appartements de moins de 90 m²**, puis lui demande d'estimer des
biens de plus de 150 m² — des surfaces qu'il n'a jamais vues.

In [ ]:
petits = ventes.query("surface < 90")
grands = ventes.query("surface > 150")

Xp = pd.concat([petits[["surface", "pieces"]],
                pd.get_dummies(petits["arrondissement"], prefix="arr", drop_first=True)], axis=1)
Xg = pd.concat([grands[["surface", "pieces"]],
                pd.get_dummies(grands["arrondissement"], prefix="arr", drop_first=True)], axis=1)
Xg = Xg.reindex(columns=Xp.columns, fill_value=False)   ## memes colonnes des deux cotes

myope = LinearRegression().fit(Xp, petits["prix"])
sur_les_grands = myope.predict(Xg)

print("sur son terrain (moins de 90 m2) :", round(mean_absolute_error(petits["prix"], myope.predict(Xp))), "euros d'erreur")
print("sur les grands (plus de 150 m2)  :", round(mean_absolute_error(grands["prix"], sur_les_grands)), "euros d'erreur")
print("et il se trompe toujours dans le meme sens :", round(np.mean(sur_les_grands - grands["prix"])), "euros")

**897 285 € d'erreur au lieu de 83 018 €**, et une sous-estimation
systématique de 699 945 €. Le modèle ne prévient pas. Il ne sait
pas qu'on lui pose une question hors de son domaine : **il répond, c'est
tout.**

---

## Partie 4 — Noter trois règles, et les départager

Vous avez un modèle et sa note. Il manque la seule chose qui compte pour la
direction : **est-ce mieux que ce qu'on a déjà ?**

Un modèle ne se juge jamais dans l'absolu. Il se juge contre ce qu'il
remplace.

### Une mesure ne regarde que deux colonnes de nombres

Relisez la forme d'un appel à une mesure :

```python
mean_absolute_error(y_test, pred_test)
```

Deux suites de nombres de même longueur : **ce qui s'est passé**, et **ce
qu'on avait annoncé**. La fonction ne sait pas d'où vient la seconde, et n'a
pas à le savoir. Elle peut venir d'un modèle scikit-learn, d'une formule
écrite à la main, ou d'un expert qui aurait donné son avis sur chaque bien.

> **Si on peut aligner un prix annoncé en face de chaque prix réel, on peut
> noter.** On ne compare pas des algorithmes : on compare des colonnes de
> prédictions.

C'est ce qui rend possible tout ce qui suit.

### Exercice 10 — La règle paresseuse

Fabriquez la prédiction la plus bête qui soit : **le prix moyen du jeu
d'apprentissage**, répété pour chacune des 6 303 ventes de test. Rangez-la
dans `paresseuse`, puis notez-la avec les trois mesures.

> **Rappel.** `np.full(combien, quelle_valeur)` fabrique une suite de nombres
> tous identiques. La moyenne à répéter est celle de `yg_train` — surtout pas
> celle du test, qu'on n'a pas le droit de regarder.

In [ ]:
verifier("la regle paresseuse", round(mean_absolute_error(yg_test, paresseuse)) == 358922,
         "np.full(len(yg_test), yg_train.mean())")
verifier("son R2", abs(r2_score(yg_test, paresseuse)) < 0.001, "il doit tomber sur zero, ou tout pres")

**R² = -0,000014.** Ce zéro n'est pas un hasard : la formule du R² compare toute
prédiction à *exactement* celle-là, la moyenne. Annoncer toujours la moyenne,
c'est la définition du zéro.

Donc **un R² se lit « de combien je fais mieux que la règle paresseuse »**, et
le modèle du consultant, à 0,791, fait 79 % du chemin.

Reste à expliquer pourquoi ce n'est pas zéro tout rond, et la réponse vaut le
détour : le R² se calcule contre la moyenne du jeu sur lequel on le mesure,
celle du **test** (565 834 €). Notre règle, elle, annonce la moyenne de
l'**apprentissage** (563 601 €), parce qu'elle n'a pas le droit de
regarder le test. Deux mille euros d'écart, et le R² passe juste sous zéro.

En annonçant la moyenne du test, on obtiendrait exactement 0,000. Autrement
dit : **même la règle la plus bête du monde peut tricher, et ici tricher se
voit à la quatrième décimale.**

### L'estimateur maison est un modèle, lui aussi

Voici ce que vous aviez écrit au bloc 3, en une ligne :

```python
prix_annonce = surface * mediane_du_prix_m2[arrondissement]
```

Trois questions, dans l'ordre :

1. **Y a-t-il un `.fit()` là-dedans ?** Non.
2. **Cette formule contient-elle des nombres qu'il a fallu calculer sur des
   données ?** Oui : **vingt médianes**, une par arrondissement.
3. **Alors, est-ce un modèle ?** Oui.

Un modèle est une règle qui transforme des caractéristiques en prédiction, et
dont les nombres viennent des données. L'estimateur maison en a vingt, celui
du consultant en a vingt et un. Ce qui les sépare n'est pas d'apprendre ou
non, c'est **la forme** : l'un **multiplie** une surface par un prix au m²,
l'autre **additionne** des euros. Souvenez-vous du biais de l'exercice 8.

### Exercice 11 — Reconstruire l'estimateur maison

Avant d'écrire, une question. Répondez en commentaire dans la cellule
suivante :

> *Les vingt médianes, faut-il les calculer sur le fichier entier ou sur le
> seul jeu d'apprentissage ? On les a toutes sous la main.*

In [ ]:
# Ma reponse (fichier entier, ou apprentissage seulement ?) :

Construisez `medianes_app` — la médiane de `prix_m2` par arrondissement,
calculée **sur les seules ventes d'apprentissage** — puis `maison`, la
prédiction de l'estimateur pour chaque vente du jeu de test.

> **Rappel.** `ventes.loc[Xg_train.index]` garde les lignes d'apprentissage.
> `groupby(...)[...].median()` rend une Series indexée par arrondissement, et
> `.map(cette_series)` remplace chaque numéro d'arrondissement par sa médiane.

In [ ]:
verifier("vingt medianes", len(medianes_app) == 20, "une par arrondissement")
verifier("calculees sur l'apprentissage seul", round(medianes_app.loc[19]) == 7816,
         "recalculez-les sur ventes.loc[Xg_train.index] : sinon votre estimateur a vu le jeu de test")
verifier("un prix par vente de test", len(maison) == 6303, "une prediction par ligne du jeu de test")

> ⚠️ **Pourquoi l'apprentissage seulement.** Ces vingt médianes sont les
> **paramètres** de l'estimateur maison. Les calculer sur le fichier entier,
> c'est le laisser regarder les ventes sur lesquelles on va le noter : sa note
> s'améliore, et elle devient fausse. C'est la fuite de données du cours 4.1,
> sous une forme que personne ne voit passer.
>
> **Une règle écrite à la main triche aussi bien qu'un modèle.** La frontière
> apprentissage / test ne protège pas que scikit-learn.

### Exercice 12 — Noter l'estimateur maison

Les trois mesures, sur `maison` cette fois.

> **Rappel.** Exactement les mêmes appels qu'à l'exercice 4. La fonction ne
> demande pas d'où vient la prédiction.

In [ ]:
verifier("la MAE de l'estimateur maison", round(mae_maison) == 118617,
         "si vous obtenez moins, vos medianes ont ete calculees sur le fichier entier")

### Exercice 13 — Le concours

Trois règles, une seule façon de les départager : les mêmes mesures, sur les
mêmes ventes.

Avant d'exécuter, engagez-vous une dernière fois.

In [ ]:
# Mon classement, du meilleur au pire : .......

La fonction suivante note une prédiction, quelle qu'elle soit. Exécutez-la,
puis construisez le tableau `concours` avec les trois règles.

> **Rappel.** `pd.DataFrame([dictionnaire1, dictionnaire2, ...])` fabrique un
> tableau à partir d'une liste de lignes. `.set_index("regle")` met les noms
> en étiquettes.

In [ ]:
def noter(nom, prediction):
    """Note n'importe quelle suite de prix annonces, d'ou qu'elle vienne."""
    return {"regle": nom,
            "R2": round(r2_score(yg_test, prediction), 3),
            "MAE": round(mean_absolute_error(yg_test, prediction)),
            "RMSE": round(mean_squared_error(yg_test, prediction) ** 0.5),
            "erreur mediane": round(np.median(np.abs(yg_test - prediction)))}

In [ ]:
verifier("trois regles notees", len(concours) == 3, "une ligne par appel a noter()")
verifier("la MAE de l'estimateur maison", concours.loc["l'estimateur maison", "MAE"] == 118617,
         "le tableau reprend les valeurs des exercices 10, 12 et 5")
verifier("la MAE du consultant", concours.loc["le modele du consultant", "MAE"] == 149483, "idem")

Lisez ce tableau ligne par ligne, puis colonne par colonne.

- Les deux vraies règles écrasent la paresseuse, évidemment.
- **Le modèle du consultant gagne le R² et la RMSE.** De très peu :
  0,791 contre 0,789.
- **L'estimateur maison gagne la MAE et l'erreur médiane.** De beaucoup :
  118 617 € contre 149 483 €, et 54 116 € contre 92 621 €
  sur l'erreur médiane.

**Le classement dépend de la mesure.** Ce n'est pas une bizarrerie, c'est le
cœur du métier. La RMSE élève les erreurs au carré : elle facture très cher
les hôtels particuliers de l'exercice 9, et le modèle du consultant est
justement un peu moins mauvais qu'ailleurs sur ces biens-là. La MAE et la
médiane décrivent l'estimation courante, celle d'un deux-pièces un mardi
matin.

La cellule suivante montre où chacun gagne.

In [ ]:
comparaison = pd.DataFrame({
    "la regle paresseuse": np.abs(yg_test - paresseuse),
    "l'estimateur maison": np.abs(yg_test - maison),
    "le modele du consultant": np.abs(yg_test - pred_grand),
})
comparaison["tranche"] = erreurs["tranche"]

(comparaison.groupby("tranche", observed=True).mean() / 1000).plot(
    kind="bar", figsize=(7, 4.2), color=["#cccccc", "#F2B5B3", "#2878B5"])
plt.ylabel("erreur moyenne (milliers d'euros)")
plt.xlabel("surface (m2)")
plt.title("Qui se trompe le moins ? Ca depend de la taille du bien")
plt.xticks(rotation=0)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Sous 25 m², l'estimateur maison se trompe de 37 191 € quand le
modèle en met 96 673 €.** Au-dessus de 150 m², les deux se valent.

L'explication est celle de l'exercice 8. Le marché **multiplie** une surface
par un prix au m² ; l'estimateur maison fait exactement ça. Le modèle, lui,
**additionne** : une constante de 82 593 €, tant d'euros par m², tant
par arrondissement. Sur un studio, cette constante pèse presque autant que le
bien lui-même, et le modèle plafonne.

### Exercice 14 — Répondre au consultant

Deux réponses à écrire en commentaire.

1. *Quelle mesure choisissez-vous pour départager, et pourquoi ?*
2. *Que répondez-vous au cabinet de conseil et à sa facture de 40 000 € ?*

Il n'y a pas une seule bonne réponse, mais il y a des réponses mal fondées :
celles qui ne citent aucun chiffre, et celles qui citent un chiffre sans dire
sur quelles ventes il a été calculé.

### Pour aller plus loin — donner au modèle ce que vous savez

Une dernière cellule, et elle renverse le résultat. On ajoute aux variables
du modèle **la prédiction de l'estimateur maison** : une colonne de plus,
`surface × médiane de l'arrondissement`.

In [ ]:
valeur_reperee = (ventes["surface"]
                  * ventes["arrondissement"].map(medianes_app)).rename("valeur_reperee")
X_plus = pd.concat([X_grand, valeur_reperee], axis=1)

Xp_train, Xp_test = X_plus.loc[Xg_train.index], X_plus.loc[Xg_test.index]
augmente = LinearRegression().fit(Xp_train, yg_train).predict(Xp_test)

pd.DataFrame([noter("le modele du consultant", pred_grand),
              noter("l'estimateur maison", maison),
              noter("le modele + la variable metier", augmente)]).set_index("regle")

**R² 0,822, RMSE 252 834 €** : le meilleur des trois sur ces deux
mesures, et la MAE se rapproche de l'estimateur maison.

Ce n'est pas l'algorithme qui a progressé — c'est la même `LinearRegression`.
C'est la **variable** qu'on lui a donnée, et cette variable, ce sont six
lignes de pandas écrites au bloc 3.

> **La phrase à retenir de cette partie :** quand un modèle plafonne, on
> gagne presque toujours plus à lui apporter une bonne variable qu'à changer
> d'algorithme.

---

## Partie 5 — Les deux pièges

### Exercice 15 — La fuite

Reprenez le modèle simple, et ajoutez `prix_m2` aux variables explicatives.
Mesurez le R² de test.

> **Rappel.** `ventes[["surface", "prix_m2"]]` pour les deux colonnes, puis
> le découpage habituel avec `random_state=67`, `fit`, `predict`, `r2_score`.

In [ ]:
verifier("le R2 de la fuite", round(r2_fuite, 3) == 0.897, "ajoutez prix_m2 a surface, rien d'autre")

**0,897 contre 0,791.** Dix points de R² gagnés d'un coup, sans
rien changer d'autre.

En commentaire : *pourquoi ce modèle est-il inutilisable le jour où un client
appelle ?*

> Notez la différence avec le cours : le score ne monte pas à 1,000. Il monte
> à 0,897. **Une fuite ne s'annonce pas toujours par un score
> parfait** — souvent, elle ressemble juste à un beau résultat.

### Exercice 16 — Le surapprentissage

`rue` était dans vos colonnes utilisables : c'est une information réelle sur
le bien, et le nom de la rue en dit long sur le prix. Essayons.

La cellule suivante fabrique une colonne 0/1 **par rue** et ajuste le modèle.

> ⏳ **Comptez une trentaine de secondes.** Ce n'est pas un bug : le modèle a
> 3 031 colonnes à estimer au lieu de 21. Le temps d'attente fait
> partie de la leçon.

In [ ]:
rues = pd.get_dummies(ventes["rue"], prefix="rue", drop_first=True)
X_rue = pd.concat([X_grand, rues], axis=1)
print(X_rue.shape[1], "colonnes pour", len(Xg_train), "ventes d'apprentissage")

Xr_train, Xr_test = X_rue.loc[Xg_train.index], X_rue.loc[Xg_test.index]
avec_rue = LinearRegression().fit(Xr_train, yg_train)

Calculez maintenant le R² **d'apprentissage** et le R² **de test** de ce
modèle, et comparez-les à ceux du modèle à 21 colonnes.

> **Rappel.** `r2_score(yg_train, avec_rue.predict(Xr_train))` pour
> l'apprentissage, et la même chose avec `_test` pour le test.

In [ ]:
verifier("le R2 d'apprentissage avec la rue", round(r2_rue_train, 2) == 0.88, "r2_score sur Xr_train")
verifier("le R2 de test avec la rue", round(r2_rue_test, 2) == 0.62, "r2_score sur Xr_test")

| | R² apprentissage | R² test |
|---|---:|---:|
| sans la rue (21 colonnes) | 0,817 | **0,791** |
| avec la rue (3 031 colonnes) | **0,88** | **0,62** |

**L'apprentissage monte, le test s'effondre.** Le modèle est devenu bien
meilleur sur les ventes qu'il a vues, et bien pire sur les autres.

En commentaire : *a-t-il appris, ou a-t-il retenu ?* Le chiffre qui donne la
réponse est dans la cellule précédente : 3 011 rues différentes pour
18 906 ventes d'apprentissage.

---

## Partie 6 — Une dernière question

### Exercice 17 — Changer de cible

Gardez exactement le même `X_grand`, les mêmes découpages, le même modèle. Ne
changez qu'une chose : au lieu du **prix**, demandez-lui de prédire le **prix
au m²**.

> **Rappel.** `y2 = ventes["prix_m2"]`, puis le découpage avec le même
> `random_state=67`, `fit`, `predict`, `r2_score`.

In [ ]:
verifier("le R2 sur le prix au m2", round(r2_m2, 3) == 0.277, "meme X_grand, meme random_state, seul y change")

**0,791 pour le prix, 0,277 pour le prix au m².** Même fichier, même
code, même jour.

En commentaire : *pourquoi ?*

Ce n'est pas un défaut du modèle, ni de la régression linéaire. **Ce que le
fichier ne contient pas, aucun modèle ne l'inventera.**

Retenez cette phrase : l'exercice suivant est entièrement construit dessus.

---

## Pour conclure

### La note à la direction

Complétez cette cellule de texte en trois phrases, avec vos chiffres :

- Nous recommandons de … Mesuré sur … ventes que le modèle n'avait jamais vues, …
- L'estimateur ne doit pas servir pour … (votre seuil de la partie 3)
- Le prochain gain viendra de … (relisez la fin de la partie 4)

### Ce que vous avez fait

- vous avez entraîné un modèle, et vous avez surtout regardé **où** il se trompe : 139 prix négatifs, -42,5 % sur les studios, des millions sur les hôtels particuliers ;
- vous avez noté trois règles avec les mêmes mesures, et découvert que **le classement dépend de la mesure choisie** ;
- vous avez vu qu'une règle écrite à la main peut tricher exactement comme un modèle ;
- vous avez fait monter un R² de dix points avec une fuite, et effondrer un R² de test avec 3 000 colonnes ;
- et vous avez répondu à un consultant avec des chiffres.

| Vous avez utilisé | Pour |
|---|---|
| `train_test_split(..., random_state=67)` | noter sur des ventes jamais vues, et toujours les mêmes |
| `LinearRegression().fit()` / `.predict()` | le modèle du consultant |
| `mean_absolute_error`, `mean_squared_error ** 0.5`, `r2_score` | les trois notes, sur n'importe quelle prédiction |
| `np.full()` | la règle paresseuse, et la définition du R² |
| `groupby().median()` + `.map()` | l'estimateur maison, reconstruit proprement |
| `pd.get_dummies(..., drop_first=True)` | l'arrondissement, puis la rue de trop |
| `pd.cut()` + `groupby().median()` | le biais, tranche de surface par tranche de surface |
| `nlargest()` | les cinq ventes que le modèle rate le plus |

### Les quatre phrases à retenir

1. **Un modèle se juge contre ce qu'il remplace**, jamais dans l'absolu. Sans
   la règle paresseuse et l'estimateur maison, « R² = 0,791 » ne veut rien
   dire.
2. **Le classement dépend de la mesure**, et la mesure se choisit selon ce que
   coûte l'erreur — pas l'inverse.
3. **Les scores ne regardent pas les prédictions une par une.** 139 prix
   négatifs ne font bouger aucune des trois mesures.
4. **Ce que le fichier ne contient pas, aucun modèle ne l'inventera.**

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.